[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/claude-code/02-mcp-servers.ipynb)

# MCP Servers personalizados — Notebook interactivo

Implementamos y probamos un servidor MCP en Python.
Veremos cómo Claude interactúa con herramientas externas a través del protocolo MCP.

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()
print("Cliente listo")
print("Nota: pip install mcp para crear servidores MCP reales")

## 1. Simular un servidor MCP en Python

Un servidor MCP expone herramientas que Claude puede llamar.
Aquí simulamos el dispatcher que ejecutaría esas herramientas.

In [ ]:
# Base de datos simulada (en producción: PostgreSQL, MongoDB, etc.)
BD_CLIENTES = [
    {"id": "c001", "nombre": "ACME Corp", "email": "admin@acme.com", "plan": "Business", "mrr": 990, "activo": True},
    {"id": "c002", "nombre": "StartupXYZ", "email": "cto@startupxyz.io", "plan": "Pro", "mrr": 299, "activo": True},
    {"id": "c003", "nombre": "FreelanceAna", "email": "ana@freelance.es", "plan": "Starter", "mrr": 29, "activo": False},
    {"id": "c004", "nombre": "MegaRetail SA", "email": "tech@megaretail.com", "plan": "Business", "mrr": 1980, "activo": True},
]

BD_SOPORTE = [
    {"id": "t001", "cliente_id": "c001", "asunto": "API timeout", "estado": "abierto", "prioridad": "alta"},
    {"id": "t002", "cliente_id": "c002", "asunto": "Problema con exportación CSV", "estado": "cerrado", "prioridad": "media"},
    {"id": "t003", "cliente_id": "c004", "asunto": "Solicitud de feature: SSO", "estado": "abierto", "prioridad": "baja"},
]

# Herramientas que expone el servidor MCP
MCP_TOOLS = [
    {
        "name": "buscar_clientes",
        "description": "Busca clientes por nombre, email o plan. Devuelve lista de coincidencias.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Término de búsqueda"},
                "plan": {"type": "string", "enum": ["Starter", "Pro", "Business"], "description": "Filtrar por plan"}
            }
        }
    },
    {
        "name": "obtener_metricas",
        "description": "Obtiene métricas clave del negocio: MRR total, clientes activos, tickets abiertos",
        "input_schema": {
            "type": "object",
            "properties": {
                "segmento": {"type": "string", "enum": ["todos", "Starter", "Pro", "Business"]}
            }
        }
    },
    {
        "name": "tickets_abiertos",
        "description": "Lista los tickets de soporte abiertos, opcionalmente filtrados por prioridad",
        "input_schema": {
            "type": "object",
            "properties": {
                "prioridad": {"type": "string", "enum": ["alta", "media", "baja"]}
            }
        }
    }
]

def ejecutar_herramienta_mcp(nombre: str, params: dict) -> str:
    """Dispatcher del servidor MCP — equivale al @app.call_tool() decorator."""
    
    if nombre == "buscar_clientes":
        query = params.get("query", "").lower()
        plan = params.get("plan")
        resultados = BD_CLIENTES
        if query:
            resultados = [c for c in resultados if query in c["nombre"].lower() or query in c["email"].lower()]
        if plan:
            resultados = [c for c in resultados if c["plan"] == plan]
        return json.dumps(resultados, ensure_ascii=False)
    
    elif nombre == "obtener_metricas":
        segmento = params.get("segmento", "todos")
        clientes = BD_CLIENTES if segmento == "todos" else [c for c in BD_CLIENTES if c["plan"] == segmento]
        activos = [c for c in clientes if c["activo"]]
        mrr_total = sum(c["mrr"] for c in activos)
        tickets_abiertos = len([t for t in BD_SOPORTE if t["estado"] == "abierto"])
        return json.dumps({
            "segmento": segmento, "clientes_total": len(clientes),
            "clientes_activos": len(activos), "mrr_total_eur": mrr_total,
            "tickets_abiertos": tickets_abiertos,
            "arr_estimado_eur": mrr_total * 12
        })
    
    elif nombre == "tickets_abiertos":
        prioridad = params.get("prioridad")
        tickets = [t for t in BD_SOPORTE if t["estado"] == "abierto"]
        if prioridad:
            tickets = [t for t in tickets if t["prioridad"] == prioridad]
        return json.dumps(tickets, ensure_ascii=False)
    
    return json.dumps({"error": f"Herramienta '{nombre}' no implementada"})

# Test directo de herramientas
print("Test de herramientas MCP:")
print(ejecutar_herramienta_mcp("obtener_metricas", {"segmento": "todos"}))

## 2. Claude usando las herramientas MCP (bucle agéntico)

In [ ]:
def claude_con_mcp(pregunta: str, max_iter: int = 5) -> str:
    """Claude con acceso a las herramientas del servidor MCP."""
    mensajes = [{"role": "user", "content": pregunta}]
    herramientas_usadas = []
    
    for _ in range(max_iter):
        resp = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=800,
            system="""Eres un asistente de negocio con acceso a los datos de la empresa.
Usa las herramientas MCP disponibles para obtener datos reales antes de responder.
Sé concreto y da cifras exactas cuando estén disponibles.""",
            tools=MCP_TOOLS,
            messages=mensajes
        )
        
        mensajes.append({"role": "assistant", "content": resp.content})
        
        if resp.stop_reason == "end_turn":
            texto = next((b.text for b in resp.content if hasattr(b, "text")), "")
            print(f"  Herramientas usadas: {herramientas_usadas}")
            return texto
        
        if resp.stop_reason == "tool_use":
            resultados = []
            for bloque in resp.content:
                if bloque.type == "tool_use":
                    herramientas_usadas.append(bloque.name)
                    print(f"  → MCP: {bloque.name}({bloque.input})")
                    resultado = ejecutar_herramienta_mcp(bloque.name, bloque.input)
                    resultados.append({"type": "tool_result", "tool_use_id": bloque.id, "content": resultado})
            mensajes.append({"role": "user", "content": resultados})
    
    return "Límite de iteraciones alcanzado"

# Consulta al servidor MCP
print("PREGUNTA 1: Estado del negocio")
print("=" * 50)
resp = claude_con_mcp("Dame un resumen del estado del negocio: MRR, clientes activos y tickets urgentes pendientes.")
print(f"\n{resp}")

In [ ]:
print("PREGUNTA 2: Análisis de clientes Business")
print("=" * 50)
resp2 = claude_con_mcp(
    "¿Cuántos clientes tenemos en el plan Business? "
    "¿Alguno tiene tickets de soporte abiertos con prioridad alta?"
)
print(f"\n{resp2}")

## 3. Esqueleto de servidor MCP real

Código listo para ejecutar como proceso independiente.

In [ ]:
CODIGO_SERVIDOR_MCP = '''
# mcp_empresa.py
# pip install mcp
# Ejecutar: python mcp_empresa.py
# Conectar en .claude/settings.json:
# {"mcpServers": {"empresa": {"command": "python", "args": ["./mcp_empresa.py"]}}}

from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent
import asyncio, json

app = Server("empresa-mcp")

@app.list_tools()
async def tools():
    return [
        Tool(
            name="buscar_clientes",
            description="Busca clientes en la base de datos",
            inputSchema={
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                }
            }
        )
    ]

@app.call_tool()
async def call(name: str, arguments: dict):
    if name == "buscar_clientes":
        # Aquí conectarías con tu BD real
        resultado = [{"id": "c001", "nombre": "ACME Corp"}]
        return [TextContent(type="text", text=json.dumps(resultado))]

async def main():
    async with stdio_server() as (r, w):
        await app.run(r, w, app.create_initialization_options())

asyncio.run(main())
'''

print("Esqueleto de servidor MCP (guarda como mcp_empresa.py):")
print(CODIGO_SERVIDOR_MCP)

# Guardar el archivo para referencia
with open("/tmp/mcp_empresa_ejemplo.py", "w") as f:
    f.write(CODIGO_SERVIDOR_MCP)
print("\nGuardado en /tmp/mcp_empresa_ejemplo.py")

## Resumen

| Componente MCP | Equivalente en este notebook |
|---|---|
| `@app.list_tools()` | `MCP_TOOLS` — lista de herramientas |
| `@app.call_tool()` | `ejecutar_herramienta_mcp()` — dispatcher |
| Protocolo stdio | `claude_con_mcp()` — bucle de tool_use |
| `.claude/settings.json` | Configuración del CLI (no en notebook) |

**Siguiente paso:** Instalar `mcp` con `pip install mcp` y ejecutar el servidor real
conectado a tu base de datos o API interna.